In [1]:
import sys
import os
import re
import tree_sitter

from src.core.tree_sitter_engine import UniversalASTReader

# Initiate the reader - UniversalASTReader with Python/Java
reader = UniversalASTReader()
print("✅ Đã nạp UniversalASTReader thành công.")


✅ Đã nạp UniversalASTReader thành công.


In [2]:
import tree_sitter

def code_to_cst_sexp(node: tree_sitter.Node, source_bytes: bytes) -> str:
    """
    Chuyển đổi một node của Tree-sitter và các con của nó thành chuỗi S-expression.
    
    Args:
        node: Đối tượng Node từ tree-sitter.
        source_bytes: Mảng bytes của mã nguồn gốc để trích xuất text.
        
    Returns:
        str: Chuỗi S-expression đại diện cho cây CST, bao gồm cả khoảng trắng (gaps).
    """
    # Trường hợp Node lá (Leaf Node): Không có con
    if not node.children:
        # Trích xuất text gốc dựa trên byte offset
        text: str = source_bytes[node.start_byte:node.end_byte].decode('utf-8', errors='replace')
        return f"({node.type} {repr(text)})"
    
    parts: list[str] = []
    current_byte: int = node.start_byte
    
    # Sắp xếp các con theo vị trí byte để đảm bảo thứ tự
    ordered_children = sorted(node.children, key=lambda x: x.start_byte)
    
    for child in ordered_children:
        # Kiểm tra và lấy phần văn bản nằm giữa các node (khoảng trắng, xuống dòng, comment ngoài cây)
        if child.start_byte > current_byte:
            gap_text: str = source_bytes[current_byte:child.start_byte].decode('utf-8', errors='replace')
            parts.append(f"(gap {repr(gap_text)})")
        
        # Đệ quy xuống các node con
        parts.append(code_to_cst_sexp(child, source_bytes))
        current_byte = child.end_byte
        
    # Lấy phần văn bản thừa cuối cùng (nếu có) bên trong node cha
    if node.end_byte > current_byte:
        final_gap: str = source_bytes[current_byte:node.end_byte].decode('utf-8', errors='replace')
        parts.append(f"(gap {repr(final_gap)})")
        
    return f"({node.type} {' '.join(parts)})"

In [3]:
def cst_sexp_to_code(sexp_str: str) -> str:
    """
    Tái tạo mã nguồn từ chuỗi S-expression bằng cách trích xuất dữ liệu từ các node lá.
    
    Args:
        sexp_str: Chuỗi S-expression đã được đóng gói.
        
    Returns:
        str: Mã nguồn hoàn chỉnh với định dạng gốc.
    """
    # Regex trích xuất nội dung trong nháy của các node (loại_node 'nội_dung')
    # Hỗ trợ mọi loại ký tự cho loại_node (kể cả symbols)
    pattern: str = r'\(([^ ]+) (("(?:\\.|[^"\\])*")|(\'(?:\\.|[^\'\\])*\'))\)'
    
    matches: list[tuple] = re.findall(pattern, sexp_str)
    code_parts: list[str] = []
    
    for _, full_match, _, _ in matches:
        try:
            # Giải mã chuỗi (un-repr) để lấy text nguyên bản
            content: str = eval(full_match)
            code_parts.append(content)
        except Exception:
            continue
        
    return "".join(code_parts)

In [4]:
PYTHON_EXAMPLE = """
a = 50
b = 10
c = a + b
if a % 2 == 0:
    print(f"{c} is an even number")
else:
    print(f"{c} is an odd number")
"""

JAVA_CODE = """
public class Main {
    public static void main(String[] args) {
        int a = 50;
        int b = 10;
        int c = a + b;

        // Check condition block
        if (a % 2 == 0) {
            System.out.println(c + " is an even number");
        } else {
            System.out.println(c + " is an odd number");
        }
    }
}
"""

input_code = PYTHON_EXAMPLE

In [5]:
source_bytes = input_code.encode('utf-8')
tree = reader.parse(source_bytes, "python")

def print_tree(node: tree_sitter.Node, source_bytes: bytes, level: int = 0, last: bool = True, prefix: str = "") -> None:
    """
    Pretty print the tree-like structure.
    """
    # Branches identify
    marker = "└── " if last else "├── "
    
    # Taking texts and leaf nodes
    text_info = ""
    if not node.children:
        text = source_bytes[node.start_byte:node.end_byte].decode('utf-8', errors='replace')
        text_info = f" \033[94m{repr(text)}\033[0m" # Màu xanh cho nội dung
    
    # Print the current node
    print(f"{prefix}{marker}{node.type}{text_info}")
    
    # Prepare the branch prefixes for its children
    new_prefix = prefix + ("    " if last else "│   ")
    
    # Traverse children nodes
    child_count = len(node.children)
    for i, child in enumerate(node.children):
        print_tree(child, source_bytes, level + 1, i == child_count - 1, new_prefix)

print("TREE STRUCTURE IN DETAILS: ")
print_tree(tree.root_node, source_bytes)


TREE STRUCTURE IN DETAILS: 
└── module
    ├── expression_statement
    │   └── assignment
    │       ├── identifier 'a'
    │       ├── = '='
    │       └── integer '50'
    ├── expression_statement
    │   └── assignment
    │       ├── identifier 'b'
    │       ├── = '='
    │       └── integer '10'
    ├── expression_statement
    │   └── assignment
    │       ├── identifier 'c'
    │       ├── = '='
    │       └── binary_operator
    │           ├── identifier 'a'
    │           ├── + '+'
    │           └── identifier 'b'
    └── if_statement
        ├── if 'if'
        ├── comparison_operator
        │   ├── binary_operator
        │   │   ├── identifier 'a'
        │   │   ├── % '%'
        │   │   └── integer '2'
        │   ├── == '=='
        │   └── integer '0'
        ├── : ':'
        ├── block
        │   └── expression_statement
        │       └── call
        │           ├── identifier 'print'
        │           └── argument_list
        │               ├── ( '

In [6]:
sexp_data = code_to_cst_sexp(tree.root_node, source_bytes)
sexp_data

'(module (expression_statement (assignment (identifier \'a\') (gap \' \') (= \'=\') (gap \' \') (integer \'50\'))) (gap \'\\n\') (expression_statement (assignment (identifier \'b\') (gap \' \') (= \'=\') (gap \' \') (integer \'10\'))) (gap \'\\n\') (expression_statement (assignment (identifier \'c\') (gap \' \') (= \'=\') (gap \' \') (binary_operator (identifier \'a\') (gap \' \') (+ \'+\') (gap \' \') (identifier \'b\')))) (gap \'\\n\') (if_statement (if \'if\') (gap \' \') (comparison_operator (binary_operator (identifier \'a\') (gap \' \') (% \'%\') (gap \' \') (integer \'2\')) (gap \' \') (== \'==\') (gap \' \') (integer \'0\')) (: \':\') (gap \'\\n    \') (block (expression_statement (call (identifier \'print\') (argument_list (( \'(\') (string (string_start \'f"\') (interpolation ({ \'{\') (identifier \'c\') (} \'}\')) (string_content \' is an even number\') (string_end \'"\')) () \')\'))))) (gap \'\\n\') (else_clause (else \'else\') (: \':\') (gap \'\\n    \') (block (expression

In [7]:
to_code = cst_sexp_to_code(sexp_data)
print(to_code)

a = 50
b = 10
c = a + b
if a % 2 == 0:
    print(f"{c} is an even number")
else:
    print(f"{c} is an odd number")



In [8]:
import json

def get_full_node_json(node, source_bytes):
    """
    Translating from tree nodes to JSON type
    """
    res = {
        "type": node.type,
        "start": node.start_byte,
        "end": node.end_byte
    }
    if not node.children:
        res["text"] = source_bytes[node.start_byte:node.end_byte].decode('utf-8', errors='replace')
    else:
        res["children"] = [get_full_node_json(c, source_bytes) for c in node.children]
    return res

def compare_complexity(node, source_bytes, sexp_str):
    """
    Comparing characters by each formats
    """
    # 1. Original codebase
    original_size = len(source_bytes)
    
    # 2. Định dạng JSON đầy đủ (Dạng "Node thuần" hay dùng trong các API)
    full_json = json.dumps(get_full_node_json(node, source_bytes))
    json_size = len(full_json)
    
    # 3. Định dạng S-expression (Chiến thuật hiện tại)
    sexp_size = len(sexp_str)
    
    print("SO SÁNH DUNG LƯỢNG (Characters/Bytes):")
    print(f"{'Định dạng':<25} | {'Dung lượng':<12} | {'Tỷ lệ so với Gốc'}")
    print("-" * 60)
    print(f"{'1. Original Code':<25} | {original_size:<12} | 100%")
    print(f"{'2. Full JSON Node':<25} | {json_size:<12} | {json_size/original_size:.1%}")
    print(f"{'3. S-expression (Nén)':<25} | {sexp_size:<12} | {sexp_size/original_size:.1%}")
    
    print(f"\n💡 KẾT LUẬN: S-expression gọn hơn JSON khoảng \033[1m{json_size/sexp_size:.1f} lần\033[0m.")

# --- Chạy so sánh ---
# Giả sử bạn đã có sexp_data từ các bước trước
compare_complexity(tree.root_node, source_bytes, sexp_data)


SO SÁNH DUNG LƯỢNG (Characters/Bytes):
Định dạng                 | Dung lượng   | Tỷ lệ so với Gốc
------------------------------------------------------------
1. Original Code          | 117          | 100%
2. Full JSON Node         | 3749         | 3204.3%
3. S-expression (Nén)     | 1115         | 953.0%

💡 KẾT LUẬN: S-expression gọn hơn JSON khoảng 3.4 lần.
